In [1]:
import numpy as np
from datasets import load_dataset

news = load_dataset('argilla/news-summary', split= 'test')
df = news.to_pandas().sample(5000, random_state = 42)[['text', 'prediction']]
df['test'] = 'summarize: ' + df['text']
df['prediction'] = df['prediction'].map(lambda x: x[0]['text'])
train, valid, test = np.split(df.sample(frac = 1, random_state = 42),
                              [int(0.6 * len(df)), int(0.8 * len(df))])
print(train.text.iloc[0][:200])
print(train.prediction.iloc[0][:50])
print(len(train))
print(len(valid))
print(len(test))

DANANG, Vietnam (Reuters) - Russian President Vladimir Putin said on Saturday he had a normal dialogue with U.S. leader Donald Trump at a summit in Vietnam, and described Trump as civil, well-educated
Putin says had useful interaction with Trump at Vi
3000
1000
1000


c:\Users\KDS10\Documents\KDT17\17-computer_vision\.venv-torchtext-cuda\Lib\site-packages\numpy\core\fromnumeric.py:59: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [2]:
import torch
from transformers import T5Tokenizer
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence

def make_dataset(data, tokenizer, device):
    source = tokenizer(text = data.text.tolist(), padding = 'max_length',
                       max_length = 128, pad_to_max_length = True, truncation = True, return_tensors = 'pt')
    target = tokenizer(text = data.prediction.tolist(), padding = 'max_length',
                       max_length = 128, pad_to_max_length = True, truncation = True, return_tensors = 'pt')
    source_ids = source['input_ids'].squeeze().to(device)
    source_mask = source['attention_mask'].squeeze().to(device)
    target_ids = target['input_ids'].squeeze().to(device)
    target_mask = target['attention_mask'].squeeze().to(device)
    return TensorDataset(source_ids, source_mask, target_ids, target_mask)

def get_dataloader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)
    dataloader = DataLoader(dataset,
                            sampler=data_sampler,
                            batch_size=batch_size)
    return dataloader

epochs = 5
batch_size = 8

device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = T5Tokenizer.from_pretrained(pretrained_model_name_or_path= 't5-small')

train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(train_dataset,
                                  RandomSampler,
                                  batch_size)

valid_dataset = make_dataset(valid,
                             tokenizer,
                             device)
valid_dataloader = get_dataloader(valid_dataset,
                                  RandomSampler,
                                  batch_size)

test_dataset = make_dataset(test,
                            tokenizer,
                            device)
test_dataloader = get_dataloader(test_dataset,
                                 RandomSampler,
                                 batch_size)

print(next(iter(train_dataloader)))
print(tokenizer.convert_ids_to_tokens(21603))
print(tokenizer.convert_ids_to_tokens(10))

tokenizer_config.json: 0.00B [00:00, ?B/s]

c:\Users\KDS10\Documents\KDT17\17-computer_vision\.venv-torchtext-cuda\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KDS10\.cache\huggingface\hub\models--t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


[tensor([[ 6554,   196,   683,  ...,    28,  1473,     1],
        [  549, 21337,  2365,  ...,    12,   253,     1],
        [  549, 21337,  2365,  ...,  5468,    30,     1],
        ...,
        [  549, 21337,  2365,  ...,  4492,   724,     1],
        [ 9766,  5332,  5999,  ..., 11527,    18,     1],
        [   41, 18844,    61,  ...,     5,   287,     1]], device='cuda:0'), tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]], device='cuda:0'), tensor([[ 1473,   845,  7511,  ...,     0,     0,     0],
        [ 2759,  2523,  1082,  ...,     0,     0,     0],
        [12330,  2776,  7721,  ...,     0,     0,     0],
        ...,
        [ 2523,     3,  1427,  ...,     0,     0,     0],
        [ 9652,    31,     7,  ...,     0,     0,     0],
        [  283,    29,  2295,  ...,     0,     0,     0]], device='cuda:0'), ten

In [3]:
from torch import optim
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained(pretrained_model_name_or_path='t5-small').to(device)
optimizer = optim.AdamW(model.parameters(), lr = 1e-5, eps = 1e-8)

config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [4]:
import numpy as np
from torch import nn

def calc_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis = 1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat / len(labels_flat))

def train(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for source_ids, source_mask, target_ids, target_mask in dataloader:
        decoder_input_ids = target_ids[:, :-1].contiguous()
        labels = target_ids[:, 1:].clone().detach()
        labels[target_ids[:, 1:] == tokenizer.pad_token_id] = -100

        outputs = model(input_ids = source_ids, attention_mask = source_mask,
                        decoder_input_ids = decoder_input_ids, labels = labels)
        loss = outputs.loss
        train_loss += loss.item()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss = train_loss / len(dataloader)
    return train_loss

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        val_loss = 0.0

    for source_ids, source_mask, target_ids, target_mask in dataloader:
        decoder_input_ids = target_ids[:, :-1].contiguous()
        labels = target_ids[:, 1:].clone().detach()
        labels[target_ids[:, 1:] == tokenizer.pad_token_id] = -100

        outputs = model(input_ids = source_ids, attention_mask = source_mask,
                        decoder_input_ids = decoder_input_ids, labels = labels)
        loss = outputs.loss
        val_loss += loss.item()

    val_loss = val_loss / len(dataloader)
    return val_loss

In [5]:
best_loss = 10000

for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss = evaluation(model, valid_dataloader)
    print(epoch+1, train_loss, val_loss)

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), './models/T5ForConditionalGeneration.pt')
        print('Saved!!')    

1 4.35598091506958 3.4144169178009034
Saved!!
2 3.530746463775635 3.008881540298462
Saved!!
3 3.219322036107381 2.8229591464996338
Saved!!
4 3.0673196169535317 2.728035873413086
Saved!!
5 2.954667594909668 2.6468876791000366
Saved!!


In [9]:
model.eval()
with torch.no_grad():
    for source_ids, source_mask, target_ids, traget_mask in test_dataloader:
        generated_ids = model.generate(input_ids=source_ids,
                                       attention_mask=source_mask,
                                       max_length=128,
                                       num_beams=3,
                                       repetition_penalty=2.5,
                                       length_penalty=1.0,
                                       early_stopping=True)
                                       
        for generated, target in zip(generated_ids, target_ids):
            pred = tokenizer.decode(generated, skip_special_tokens=True,
                                    clean_up_tokenization_spaces=True)
            actual = tokenizer.decode(target, skip_special_tokens=True,
                                    clean_up_tokenization_spaces=True)
            print(pred)
            print(actual)
        break

Trump's eldest son eagerly agreed last year to meet Russian lawyer, emails show. Trump campaign officials welcomed Russian help to win election
Trump Jr. emails suggest he welcomed Russian help against Clinton
: Palestinians warn the United States against abandoning two-state solution to Israel conflict. White House official says peace does not necessarily have to entail Palestinian statehood
Palestinians caution Trump against abandoning two-state concept
to move U.S. embassy to Jerusalem could have "explosive" results
Obama suggests U.S. embassy move to Jerusalem could be 'explosive'
intelligence officials urge lawmakers to give them greater legal authority to hack back in foreign cyber attacks
German spy agencies want right to destroy stolen data and 'hack back'
to implement international tribunal ruling on oilfield dispute. Ghana and Ivory Coast set up body to implement international tribunal ruling
Ghana and Ivory Coast act to implement ruling on maritime border dispute
C. (Reuters